# ✈️ Airspeed Protection Reflex Agent
### LangGraph + Groq + Gradio UI

Prevents aircraft stall using a reflex agent + LLM refinement.

In [ ]:
!pip install -U langgraph langchain langchain-core langchain-community langchain-groq gradio

In [ ]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
import gradio as gr

In [ ]:
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

In [ ]:
class FlightState(TypedDict):
    airspeed: float
    aoa: float
    altitude: float
    decision: str

STALL_SPEED_THRESHOLD = 120
CRITICAL_AOA = 15

In [ ]:
def reflex_logic(state: FlightState):
    airspeed = state["airspeed"]
    aoa = state["aoa"]

    if airspeed < STALL_SPEED_THRESHOLD:
        decision = "⚠️ Stall Risk: Pitch Nose Down Immediately"
    elif aoa > CRITICAL_AOA:
        decision = "⚠️ High AoA: Reduce Pitch"
    else:
        decision = "✅ Stable Flight: Maintain Current Control"

    return {**state, "decision": decision}

In [ ]:
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0
)

def llm_decision(state: FlightState):
    prompt = f"""
You are an aircraft safety system.

Sensor Readings:
- Airspeed: {state['airspeed']} knots
- Angle of Attack: {state['aoa']} degrees
- Altitude: {state['altitude']} ft

Rule-based Decision:
{state['decision']}

Convert this into a clear professional pilot instruction.
"""

    response = llm.invoke(prompt)
    return {**state, "decision": response.content}

In [ ]:
builder = StateGraph(FlightState)

builder.add_node("reflex", reflex_logic)
builder.add_node("llm_refine", llm_decision)

builder.set_entry_point("reflex")
builder.add_edge("reflex", "llm_refine")
builder.add_edge("llm_refine", END)

graph = builder.compile()

In [ ]:
def run_agent(airspeed, aoa, altitude):
    state = {
        "airspeed": float(airspeed),
        "aoa": float(aoa),
        "altitude": float(altitude),
        "decision": ""
    }

    result = graph.invoke(state)
    return result["decision"]

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    <h1 style="text-align:center; color:#2c3e50;">
    ✈️ Airspeed Protection Reflex Agent
    </h1>

    <marquee style="color:#3498db; font-size:18px;">
    Real-Time Aircraft Safety System
    </marquee>
    """)

    with gr.Row():
        airspeed = gr.Slider(50, 300, value=130, label="Airspeed (knots)")
        aoa = gr.Slider(0, 25, value=10, label="Angle of Attack (°)")
        altitude = gr.Slider(0, 40000, value=10000, label="Altitude (ft)")

    output = gr.Textbox(label="Control Action", lines=4)

    btn = gr.Button("Evaluate Flight Safety")
    btn.click(fn=run_agent, inputs=[airspeed, aoa, altitude], outputs=output)

app.launch()